# Multi-branch V1 experiment

Uses the `base_utils_sol` and `multibranch_attention_sol` modules (plus the updated `sequence_explorer_utils`) from `src/`. The original `multibranch_alpha.ipynb` still runs the `base_utils_qwen` and `multibranch_attention` modules from commit `93be8d9`.

### Search modes
Choose `search_mode = 'best params'`, `'evo'`, `'bayes'`, or `'grid'`. Best params mode evaluates the single `BEST_PARAMS` configuration through CV, then refits it for holdout. Edit that dictionary directly to change the fixed configuration.

Evolutionary population and generations are set in the configuration cell. The search rejects incompatible combinations before fitting and caches equivalent candidates. `n_iter` and the workflow's `--iterations` override apply to Bayesian search.

### Kaggle Workflow Commands Reminder
```powershell
# 1. Authenticate (one-time)
& .venv-kaggle\Scripts\python.exe scripts/kaggle_experiment.py auth

# 2. Prepare local build (--iterations 2 overrides Bayesian iterations only)
& .venv-kaggle\Scripts\python.exe scripts/kaggle_experiment.py prepare --notebook multibranch_v1.ipynb --attached-source
& .venv-kaggle\Scripts\python.exe scripts/kaggle_experiment.py prepare --notebook multibranch_v1.ipynb --attached-source --iterations 2

# 3. Push notebook to Kaggle GPU kernel
& .venv-kaggle\Scripts\python.exe scripts/kaggle_experiment.py push --notebook multibranch_v1.ipynb --attached-source

# 4. Monitor status, stream logs, wait, and download results
& .venv-kaggle\Scripts\python.exe scripts/kaggle_experiment.py status
& .venv-kaggle\Scripts\python.exe scripts/kaggle_experiment.py logs
& .venv-kaggle\Scripts\python.exe scripts/kaggle_experiment.py wait
& .venv-kaggle\Scripts\python.exe scripts/kaggle_experiment.py download
```

# Multi-Branch Keras â€” Modality-Specific Sequence Classifier

This notebook uses a multi-branch deep learning architecture where each sensor modality (Acceleration, Rotation, TOF, Thermopile, STFT, CWT) has its own dedicated branch. 

It follows the exact style of the Random Forest notebook:
- configuration
- data loading
- split
- estimator (Keras Multi-Branch)
- parameter search (grid, Bayesian, or evolutionary)
- holdout evaluation
- save results

In [ ]:
import os
import sys
import warnings
import json
import importlib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import tensorflow as tf

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Local path routing
current_dir = os.getcwd()
workspace_root = current_dir
if os.path.basename(current_dir) == 'notebooks':
    workspace_root = os.path.dirname(current_dir)

src_path = os.path.join(workspace_root, 'src')
sys.path.insert(0, workspace_root)
sys.path.insert(0, src_path)

# Robust Kaggle path routing (handles nested folders from zip uploads)
if os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        # If we find our custom modules in this directory, add it to sys.path
        if 'data_utils.py' in files or 'base_utils_sol.py' in files or 'multibranch_attention_sol.py' in files:
            if root not in sys.path:
                sys.path.insert(0, root)

# Prefer this run's exact runtime-patched source snapshot.
sys.path.insert(0, src_path)

from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline

try:
    import skopt
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    BayesSearchCV = None
    Categorical = Integer = Real = None
    SKOPT_AVAILABLE = False

import data_utils
import multibranch_attention_sol as multibranch_keras
importlib.reload(multibranch_keras)
from base_utils_sol import (
    SequenceExtractor,
    SensorAugmentor,
    ECat,
    EInt,
    EFloat,
    GASearchCV,
    build_quick_results_df,
    evaluate_holdout,
    prepare_bayesian_space,
    prepare_evolutionary_space,
)
from multibranch_experiment_utils import (
    candidate_score, refit_valid_candidate,
    HierarchicalGASearchCV, HierarchicalBayesSearchCV,
)
import matplotlib.pyplot as plt
from pathlib import Path

print('Imports loaded successfully')
print('GPU devices:', tf.config.list_physical_devices('GPU'))

# Keep fitting errors visible even when other library warnings are quiet.
warnings.filterwarnings('always', message='Candidate skipped:', category=RuntimeWarning)


In [ ]:
# Runtime-only safeguards; search candidates and training settings stay unchanged.
import gc
import threading
import time

for _gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(_gpu, True)
if not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('This experiment requires a GPU.')
tf.config.optimizer.set_jit(False)
_original_model_compile = getattr(tf.keras.Model.compile, '_multibranch_original', tf.keras.Model.compile)
def _compile_without_xla(self, *args, **kwargs):
    kwargs['jit_compile'] = False
    return _original_model_compile(self, *args, **kwargs)
_compile_without_xla._multibranch_original = _original_model_compile
tf.keras.Model.compile = _compile_without_xla

_runtime_root = Path(workspace_root) / 'runtime_diagnostics'
_runtime_root.mkdir(exist_ok=True)
_runtime_lock = threading.Lock()
def _runtime_sample(stage):
    record = {'time_utc': datetime.utcnow().isoformat(), 'stage': stage}
    try:
        status = Path('/proc/self/status').read_text()
        for line in status.splitlines():
            if line.startswith(('VmRSS:', 'VmHWM:')):
                key, value = line.split(':', 1)
                record[key] = value.strip()
    except OSError:
        pass
    for name in ['memory.current', 'memory.peak', 'memory.max', 'memory.events']:
        try:
            record[name] = (Path('/sys/fs/cgroup') / name).read_text().strip()
        except OSError:
            pass
    for index in range(len(tf.config.list_physical_devices('GPU'))):
        try:
            record[f'gpu{index}'] = tf.config.experimental.get_memory_info(f'GPU:{index}')
        except (ValueError, RuntimeError):
            pass
    with _runtime_lock:
        with (_runtime_root / 'memory.jsonl').open('a') as stream:
            stream.write(json.dumps(record) + '\n')
        print('[runtime]', json.dumps(record), flush=True)

def _memory_monitor():
    while True:
        time.sleep(60)
        _runtime_sample('minute')
if not globals().get('_runtime_monitor_started', False):
    threading.Thread(target=_memory_monitor, daemon=True).start()
    _runtime_monitor_started = True

_original_candidate_score = getattr(candidate_score, '_multibranch_original', candidate_score)
def candidate_score(estimator, X, y):
    score = _original_candidate_score(estimator, X, y)
    params = {key: value for key, value in estimator.get_params(deep=True).items()
              if key.startswith('estimator__')
              and key not in {'estimator__extractor', 'estimator__augmentor'}}
    record = {'score': float(score), 'rows': len(X), 'params': params,
              'time_utc': datetime.utcnow().isoformat()}
    if hasattr(X, 'columns') and 'sequence_id' in X.columns:
        record['sequences'] = int(X['sequence_id'].nunique())
    with (_runtime_root / 'candidate_scores.jsonl').open('a') as stream:
        stream.write(json.dumps(record, default=str) + '\n')
    print('[checkpoint] score=', score, 'sequences=', record.get('sequences'), flush=True)
    _runtime_sample('score')
    return score
candidate_score._multibranch_original = _original_candidate_score
_runtime_sample('startup')
print('Runtime safeguards enabled: memory growth, JIT disabled, per-score checkpoints.')


In [ ]:
experiment_name = 'multibranch_v1'

# Supported targets: 'bfrb', 'gesture_action', 'orientation', 'gesture', 'gesture_position'
TARGET_COL = 'bfrb'

# Use sample.csv for a quick smoke test; set False for full train.csv
use_sample_data = False
sample_file = 'sample.csv'

search_mode = 'evo'  # 'best params', 'evo', 'bayes', or 'grid'
quick_save = True         # True -> save only the CV CSV with best_score & holdout_score appended; False -> full save
random_state = 42
n_splits = 1              # 1 -> GroupShuffleSplit; >=2 -> GroupKFold

train_size = 0.4
n_iter = 37
verbose = 3
epoch_verbosity = 2
error_score = -1.0  # finite penalty; valid F1 scores are >= 0
n_jobs = 1                 # TensorFlow models should not be fitted in parallel

# Evolutionary controls: initial population plus the configured generations.
# Probabilities are per pair (crossover) and per parameter (mutation).
population_size = 15
generations = 3
tournament_size = 3
elitism = True
crossover_probability = 0.7
mutation_probability = 0.15

# Optional heads: enable them independently or vary their settings in the search spaces.
USE_ORIENTATION_AUX_HEAD = False
USE_PHASE_AUX_HEAD = False
ORIENTATION_LOSS_WEIGHT = 0.2
PHASE_LOSS_WEIGHT = 0.2
# Callable (partition_dataframe, extracted_chunks) -> integer array (chunks, timesteps).
# Called separately for each CV/internal-validation partition. None skips phase candidates.
PHASE_LABEL_PROVIDER = None
# Any target column constant within a sequence can be used (classification or regression).
# Window heads read the last window_size valid timesteps of the shared chunk representation.
# Window size must not exceed chunk_window_size; sizes are timesteps, not seconds.
NO_CUSTOM_AUX_HEADS = []

generic_aux_heads = [
    {'name': 'short_window', 'target': TARGET_COL, 'window_size': 50,
     'hidden_units': [32], 'dropout': 0.1, 'loss_weight': 0.2},
    {'name': 'long_window', 'target': TARGET_COL, 'window_size': 250,
     'hidden_units': [64], 'dropout': 0.1, 'loss_weight': 0.2},
    {'name': 'action_aux', 'target': 'gesture_action', 'loss_weight': 0.2},
    {'name': 'numeric_aux', 'target': 'your_numeric_column', 'task': 'regression',
     'hidden_units': [32, 16], 'loss_weight': 0.1},
]

competition_custom_aux_heads = [
    {'name': 'action_aux', 'target': 'gesture_action',
     'hidden_units': [64, 32], 'dropout': 0.2, 'loss_weight': 0.3},
    {'name': 'position_aux', 'target': 'gesture_position',
     'hidden_units': [64, 32], 'dropout': 0.2, 'loss_weight': 0.3},
]

multi_scale_custom_aux_heads = [
    {'name': 'short_window', 'target': TARGET_COL, 'window_size': 50,
     'hidden_units': [32], 'dropout': 0.1, 'loss_weight': 0.15},
    {'name': 'long_window', 'target': TARGET_COL, 'window_size': 250,
     'hidden_units': [64], 'dropout': 0.1, 'loss_weight': 0.15},
]

qwen_custom_aux_heads = [
    {'name': 'action_aux', 'target': 'gesture_action',
     'hidden_units': [64, 32], 'dropout': 0.2, 'loss_weight': 0.3},
    {'name': 'position_aux', 'target': 'gesture_position',
     'hidden_units': [64, 32], 'dropout': 0.2, 'loss_weight': 0.3},
    {'name': 'short_window', 'target': TARGET_COL, 'window_size': 50,
     'hidden_units': [32], 'dropout': 0.1, 'loss_weight': 0.15},
    {'name': 'orient_aux', 'target': 'orientation',
     'hidden_units': [32], 'dropout': 0.1, 'loss_weight': 0.2},
]

INVALID_CANDIDATE_POLICY = 'penalize'  # 'raise' for debugging

orientation_filter_list = [] # 'Seated Lean Non Dom - FACE DOWN', 'Lie on Side - Non Dominant', 'Seated Straight', 'Lie on Back'
gesture_position_filter_list = [] # 'Neck', 'Above ear', 'Cheek', 'Eyelash', 'Eyebrow', 'Forehead'
gesture_filter_list = [] # 'Cheek - pinch skin', 'Eyelash - pull hair', 'Eyebrow - pull hair', 'Forehead - scratch', 'Forehead - pull hairline'

problematic_sequence_bool = False

target_only_bool = True

results_dir = Path(workspace_root) / f'results_{experiment_name}_{TARGET_COL}_{search_mode}'
results_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

search_mode = str(search_mode).strip().lower()
if search_mode not in ('best params', 'evo', 'bayes', 'grid'):
    raise ValueError("search_mode must be 'best params', 'evo', 'bayes', or 'grid'")

if n_splits <= 1:
    cv_object = GroupShuffleSplit(
        n_splits=1,
        test_size=(1 - train_size),
        random_state=random_state,
    )
else:
    cv_object = GroupKFold(n_splits=n_splits)

In [ ]:
# Fixed configuration for best params mode.
BEST_PARAMS = {'estimator__activation': 'relu',
 'estimator__augmentor__augmentations': '[]',
 'estimator__augmentor__noise_std': 0.0,
 'estimator__augmentor__per_aug_prob': 0.0,
 'estimator__augmentor__prob': 0.0,
 'estimator__augmentor__temporal_mask_frac': 0.0,
 'estimator__augmentor__temporal_num_masks': 0,
 'estimator__augmentor__time_shift_frac': 0.0,
 'estimator__batch_size': 128,
 'estimator__branch_dense_units': 225,
 'estimator__branch_filter_mode': 'constant',
 'estimator__branch_filters': '{"acc": 128, "cwt": 16, "rotation": 128, "stft": 16, "thermo": 32, "tof": '
                              '128}',
 'estimator__branch_kernel_sizes': '{"acc": 7, "cwt": 3, "rotation": 7, "stft": 3, "thermo": 3, "tof": 7}',
 'estimator__branch_num_conv_layers': '{"acc": 4, "cwt": 4, "rotation": 4, "stft": 4, "thermo": 2, "tof": 4}',
 'estimator__branch_pool_sizes': '{"acc": 1, "cwt": 1, "rotation": 1, "stft": 1, "thermo": 1, "tof": 1}',
 'estimator__class_weight_mode': 'balanced',
 'estimator__conv_dropout': 0.0,
 'estimator__cosine_t_max': 50,
 'estimator__dense_units': 128,
 'estimator__dropout': 0.0,
 'estimator__early_stopping_patience': 20,
 'estimator__epochs': 150,
 'estimator__extractor__acc_modes': 'smoothed|velocity|displacement|jerk',
 'estimator__extractor__add_global_context': False,
 'estimator__extractor__chunk_stride': 50,
 'estimator__extractor__chunk_stride_ratio': 0.9,
 'estimator__extractor__chunk_window_size': 200,
 'estimator__extractor__clip_value': None,
 'estimator__extractor__compute_dt': False,
 'estimator__extractor__cwt_configs': '[{"max_scale": 16, "n_scales": 8, "name": "cwt_morl", "wavelet": '
                                      '"morl"}]',
 'estimator__extractor__dead_reckoning_detrend': False,
 'estimator__extractor__filter_problematic_sequences': True,
 'estimator__extractor__final_window_mode': 'overlap',
 'estimator__extractor__fix_quaternion_sign': True,
 'estimator__extractor__ideal_skew_threshold': 1.6,
 'estimator__extractor__imu_native_sampling_rate': 100,
 'estimator__extractor__imu_target_sampling_rate': 100,
 'estimator__extractor__interp_mode': 'linear',
 'estimator__extractor__kalman_measurement_noise': 0.01,
 'estimator__extractor__kalman_process_noise': 0.0001,
 'estimator__extractor__linear_acc_mode': 'baseline',
 'estimator__extractor__max_windows_per_sequence': 150,
 'estimator__extractor__maxlen': None,
 'estimator__extractor__motion_filter_mode': None,
 'estimator__extractor__output_format': 'chunks',
 'estimator__extractor__padding_value': 0.0,
 'estimator__extractor__problematic_feature_cols': '["acc_x", "acc_y", "acc_z", "rot_x", "rot_y", "rot_w", '
                                                   '"rot_z"]',
 'estimator__extractor__problematic_features_threshold': 7,
 'estimator__extractor__resample_modalities': False,
 'estimator__extractor__rot_native_sampling_rate': 100,
 'estimator__extractor__rot_target_sampling_rate': 100,
 'estimator__extractor__rotation_modes': 'quaternion|euler|angular_velocity',
 'estimator__extractor__sequence_crop_mode': 'none',
 'estimator__extractor__smooth_alpha': None,
 'estimator__extractor__stft_configs': '[{"name": "stft_default", "noverlap": 16, "nperseg": 32}]',
 'estimator__extractor__thm_modes': 'centered_diff',
 'estimator__extractor__thm_native_sampling_rate': 20,
 'estimator__extractor__thm_target_sampling_rate': 20,
 'estimator__extractor__tof_fill_mode': 'far_500',
 'estimator__extractor__tof_modes': 'pooled_stats|sensor_stats',
 'estimator__extractor__tof_n_sensors': None,
 'estimator__extractor__tof_native_sampling_rate': 20,
 'estimator__extractor__tof_target_sampling_rate': 20,
 'estimator__extractor__use_acc_magnitude': True,
 'estimator__extractor__use_chunk_stride_ratio': True,
 'estimator__extractor__use_dead_reckoning': False,
 'estimator__extractor__use_highpass_fallback': True,
 'estimator__extractor__use_linear_acc_magnitude': True,
 'estimator__extractor__window_cap_policy': 'uniform',
 'estimator__extractor__window_size': 5,
 'estimator__gradient_clip_norm': None,
 'estimator__gradient_clip_value': None,
 'estimator__label_smoothing': 0.0,
 'estimator__learning_rate': 0.0001,
 'estimator__loss_name': 'focal',
 'estimator__lr_scheduler_type': 'cosine',
 'estimator__min_lr': 1e-06,
 'estimator__momentum': 0.0,
 'estimator__optimizer_name': 'adamw',
 'estimator__pooling_mode': 'attention',
 'estimator__static_branch_units': 100,
 'estimator__static_dropout': 0.0,
 'estimator__use_batchnorm': True,
 'estimator__use_lr_scheduler': True,
 'estimator__validation_split': 0.15,
 'estimator__warmup_epochs': 0,
 'estimator__weight_decay': 0.0}


# Prep

In [ ]:
data_root = data_utils.find_data_root()
sample_path = data_root / sample_file

if use_sample_data and sample_path.exists():
    raw_train_df = pd.read_csv(sample_path)
    print(f'Using {sample_file}: {raw_train_df["sequence_id"].nunique()} sequences')
else:
    raw_train_df = pd.read_csv(data_root / 'train.csv')
    print(f'Using train.csv: {raw_train_df["sequence_id"].nunique()} sequences')

train_demo_path = data_root / 'train_demographics.csv'
train_demo_df = pd.read_csv(train_demo_path) if train_demo_path.exists() else None

# Derive all supported sequence targets using the same definitions as random_forest.ipynb.
train_df = raw_train_df.copy()
train_df['gesture'] = train_df['gesture'].fillna('non_bfrb').astype(str)
train_df['orientation'] = train_df['orientation'].fillna('Unknown').astype(str)
train_df['is_target'] = train_df['sequence_type'].eq('Target').astype(int)
train_df['bfrb'] = train_df['gesture'].where(train_df['is_target'].astype(bool), 'non_bfrb')
train_df['gesture_position'] = train_df['gesture'].str.split(' - ').str[0]
train_df['gesture_action'] = train_df['gesture'].str.split(' - ').str[-1]

if TARGET_COL not in {'bfrb', 'gesture_action', 'orientation', 'gesture', 'gesture_position'}:
    raise ValueError("TARGET_COL must be one of 'bfrb', 'gesture_action', 'orientation', or 'gesture'.")

if orientation_filter_list:
    train_df = train_df[train_df['orientation'].isin(orientation_filter_list)].copy()

if gesture_position_filter_list:
    train_df = train_df[train_df['gesture_position'].isin(gesture_position_filter_list)].copy()

if gesture_filter_list:
    train_df = train_df[train_df['gesture'].isin(gesture_filter_list)].copy()

if target_only_bool:
    train_df = train_df[train_df['is_target'] == 1].copy()

# Orientation-based features require at least one valid quaternion per sequence.
# Entirely missing orientation cannot be repaired by within-sequence interpolation.
quaternion_cols = ['rot_w', 'rot_x', 'rot_y', 'rot_z']
quaternions = train_df[quaternion_cols].to_numpy(dtype=float)
valid_quaternion = np.isfinite(quaternions).all(axis=1) & (np.linalg.norm(quaternions, axis=1) > 0)
orientation_available = pd.Series(valid_quaternion, index=train_df.index).groupby(train_df['sequence_id']).any()
excluded_orientation_ids = orientation_available.index[~orientation_available].tolist()
if excluded_orientation_ids:
    print(f'Excluding {len(excluded_orientation_ids)} sequences with no valid orientation before splitting.')
    train_df = train_df[~train_df['sequence_id'].isin(excluded_orientation_ids)].copy()
with open(results_dir / f'{timestamp}_input_exclusions.json', 'w') as stream:
    json.dump({'reason': 'no finite nonzero quaternion in sequence',
               'sequence_ids': excluded_orientation_ids,
               'evaluation_scope': 'sequences with orientation data'}, stream, indent=2)

train_df[TARGET_COL] = train_df[TARGET_COL].fillna('non_bfrb').astype(str)
print(f'Target: {TARGET_COL}; sequences after filters: {train_df["sequence_id"].nunique()}')

In [ ]:
try:
    train_sample_df, hold_out_df = data_utils.sample_balanced_split(
        train_df,
        train_pct=train_size,
        test_pct=(1 - train_size),
        random_state=random_state,
    )
except Exception as split_error:
    print(f'Balanced split unavailable ({split_error}); using grouped random split.')
    sequence_df = (
        train_df[['sequence_id', 'is_target', TARGET_COL]]
        .drop_duplicates('sequence_id')
        .sort_values('sequence_id')
    )
    splitter = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=random_state)
    train_idx, test_idx = next(splitter.split(sequence_df, groups=sequence_df['sequence_id']))
    train_ids = sequence_df.iloc[train_idx]['sequence_id']
    test_ids = sequence_df.iloc[test_idx]['sequence_id']
    train_sample_df = train_df[train_df['sequence_id'].isin(train_ids)].copy()
    hold_out_df = train_df[train_df['sequence_id'].isin(test_ids)].copy()

X_train = train_sample_df.copy()
X_test = hold_out_df.copy()
# Preserve all candidate sequence labels, including custom targets selected in search.
# Varying per-timestep columns are rejected as sequence targets by the estimator.
y_train = X_train.copy()
y_test = X_test.copy()
groups = X_train['sequence_id'].astype(str)

print('Train sequences:', X_train['sequence_id'].nunique())
print('Holdout sequences:', X_test['sequence_id'].nunique())

In [ ]:
extractor = SequenceExtractor(
    output_format='chunks',
    acc_modes='smoothed|velocity|displacement|jerk',
)

clf = multibranch_keras.MultiBranchSequenceClassifier(
    extractor=extractor,
    augmentor=SensorAugmentor(
        sequence_col='sequence_id',
        counter_col='sequence_counter',
        prob=0.0,
        per_aug_prob=0.0,
    ),
    primary_target=TARGET_COL,
    use_orientation_aux_head=USE_ORIENTATION_AUX_HEAD,
    use_phase_aux_head=USE_PHASE_AUX_HEAD,
    custom_aux_heads=NO_CUSTOM_AUX_HEADS,
    orientation_loss_weight=ORIENTATION_LOSS_WEIGHT,
    phase_loss_weight=PHASE_LOSS_WEIGHT,
    phase_label_provider=PHASE_LABEL_PROVIDER,
    invalid_candidate_policy=INVALID_CANDIDATE_POLICY,
    candidate_failure_log=str(results_dir / f'{timestamp}_candidate_failures.jsonl'),
    verbose=epoch_verbosity,
    random_state=random_state,
)

pipeline = Pipeline([
    ('estimator', clf),
])

# Params

In [ ]:
# ============================================================
# Branch filter / depth candidates (unchanged)
# ============================================================

SEARCH_BRANCH_FILTERS = [
    {'acc': 64, 'rotation': 64, 'tof': 64, 'thermo': 32, 'stft': 64, 'cwt': 64},
    {'acc': 128, 'rotation': 128, 'tof': 128, 'thermo': 32, 'stft': 128, 'cwt': 128},
    {'acc': 160, 'rotation': 160, 'tof': 160, 'thermo': 32, 'stft': 160, 'cwt': 160},
]

SEARCH_BRANCH_DEPTHS = [
    # {'acc': 2, 'rotation': 2, 'tof': 2, 'thermo': 1, 'stft': 1, 'cwt': 1},
    # {'acc': 2, 'rotation': 2, 'tof': 2, 'thermo': 1, 'stft': 2, 'cwt': 2},
    # {'acc': 3, 'rotation': 3, 'tof': 2, 'thermo': 1, 'stft': 1, 'cwt': 2},
    {'acc': 4, 'rotation': 4, 'tof': 4, 'thermo': 2, 'stft': 4, 'cwt': 4},
]

SEARCH_CUSTOM_FILTERS = [
    {
        'acc': [32, 64, 64],
        'rotation': [32, 64, 64],
        'tof': [32, 64],
        'thermo': [32],
        'stft': [32],
        'cwt': [32],
    },
]

SEARCH_CONSTANT_FILTERS = [
    # {'acc': 16, 'rotation': 16, 'tof': 16, 'thermo': 16, 'stft': 16, 'cwt': 16},
    {'acc': 32, 'rotation': 32, 'tof': 32, 'thermo': 32, 'stft': 16, 'cwt': 16},
    # {'acc': 64, 'rotation': 64, 'tof': 64, 'thermo': 32, 'stft': 32, 'cwt': 32},
    {'acc': 128, 'rotation': 128, 'tof': 128, 'thermo': 32, 'stft': 16, 'cwt': 16},
    # {'acc': 128, 'rotation': 128, 'tof': 128, 'thermo': 32, 'stft': 32, 'cwt': 32},
    # {'acc': 167, 'rotation': 167, 'tof': 167, 'thermo': 16, 'stft': 64, 'cwt': 64},
    #{'acc': 200, 'rotation': 200, 'tof': 200, 'thermo': 32, 'stft': 64, 'cwt': 64},
]

branch_filter_mode = 'constant'  # 'custom', 'double', or 'constant'

if branch_filter_mode == 'custom':
    branch_filter_candidates = SEARCH_CUSTOM_FILTERS
    branch_depth_candidates = [{
        'acc': 3, 'rotation': 3, 'tof': 2,
        'thermo': 1, 'stft': 1, 'cwt': 1,
    }]
else:
    if branch_filter_mode == 'double':
        branch_filter_candidates = SEARCH_BRANCH_FILTERS
    elif branch_filter_mode == 'constant':
        branch_filter_candidates = SEARCH_CONSTANT_FILTERS
    else:
        raise ValueError("branch_filter_mode must be 'custom', 'double', or 'constant'.")
    branch_depth_candidates = SEARCH_BRANCH_DEPTHS


# ============================================================
# Acquisition-rate assumptions (configurable; not verified hardware metadata).
IMU_ASSUMED_FS_HZ = 100.0
ROT_ASSUMED_FS_HZ = 100.0
TOF_ASSUMED_FS_HZ = 20.0
THERMO_ASSUMED_FS_HZ = 20.0

# STFT / CWT config-list candidates
# ============================================================

STFT_CONFIGS = [
    #  None,
    # [{'name': 'stft_default', 'nperseg': 8, 'noverlap': 4}],
    # [{'name': 'stft_default', 'nperseg': 16, 'noverlap': 8}],
    [{'name': 'stft_default', 'window_seconds': 1.0, 'noverlap': 50}],
    # [{'name': 'stft_default', 'nperseg': 64, 'noverlap': 32}],
    # [
    #     {'name': 'stft_fast', 'nperseg': 32,  'noverlap': 16},
    #     {'name': 'stft_mid',  'nperseg': 64,  'noverlap': 32},
    #     {'name': 'stft_slow', 'nperseg': 128, 'noverlap': 64},
    # ],
]

CWT_CONFIGS = [
    #  None,
    # [{'name': 'cwt_morl', 'wavelet': 'morl', 'max_scale': 8, 'n_scales': 4}],
    [{'name': 'cwt_morl', 'wavelet': 'morl', 'max_scale': 16, 'n_scales': 8}],
    # [{'name': 'cwt_morl', 'wavelet': 'morl', 'max_scale': 64, 'n_scales': 32}],
    # [
    #     {'name': 'cwt_morl',  'wavelet': 'morl',  'max_scale': 64, 'n_scales': 32},
    #     {'name': 'cwt_mexh',  'wavelet': 'mexh',  'max_scale': 64, 'n_scales': 32},
    #     {'name': 'cwt_gaus2', 'wavelet': 'gaus2', 'max_scale': 64, 'n_scales': 32},
    # ],
]


# ============================================================
# FULL GRID / BAYESIAN PARAM SPACE
# ============================================================

GRID_PARAM_SPACE = {

    # ----------------------------------------------------------
    # AUGMENTOR (disabled; applied only inside estimator.fit)
    # ----------------------------------------------------------
    'estimator__augmentor__prob':                [0.0],
    'estimator__augmentor__per_aug_prob':        [0.0],
    'estimator__augmentor__augmentations':       [None],
    'estimator__augmentor__sensor_drop_prob':    [0.0],
    'estimator__augmentor__noise_std':           [0.0],
    'estimator__augmentor__jitter_sigma':        [0.0],
    'estimator__augmentor__scaling_sigma':       [0.0],
    'estimator__augmentor__channel_drop_prob':   [0.0],
    'estimator__augmentor__time_shift_frac':     [0.0],
    'estimator__augmentor__temporal_num_masks':  [0],
    'estimator__augmentor__temporal_mask_frac':  [0.0],
    'estimator__augmentor__warp_sigma':          [0.0],
    'estimator__augmentor__warp_num_knots':      [0],
    'estimator__augmentor__crop_frac_range':     [(1.0, 1.0)],

    # ----------------------------------------------------------
    # EXTRACTOR â€” accelerometer
    # ----------------------------------------------------------
    'estimator__extractor__acc_modes': [
        'smoothed|velocity|displacement|jerk',
        # 'raw|smoothed|velocity|displacement|jerk',
        # 'smoothed|jerk',
        # 'raw|smoothed|jerk',
        # 'smoothed|velocity|jerk',
        # None,
    ],
    'estimator__extractor__use_acc_magnitude':         [True],
    'estimator__extractor__use_linear_acc_magnitude':  [True],
    'estimator__extractor__linear_acc_mode':           [
        # None, 
                                                         'baseline'
                                                       ],
    'estimator__extractor__use_highpass_fallback':     [True],

    # ----------------------------------------------------------
    # EXTRACTOR â€” rotation
    # ----------------------------------------------------------
    'estimator__extractor__rotation_modes': [
        # None,
        'quaternion|euler|angular_velocity',
        # 'euler|angular_velocity',
        # 'quaternion',
        # 'rot6d',
        # 'raw'
    ],
    'estimator__extractor__fix_quaternion_sign': [True],

    # ----------------------------------------------------------
    # EXTRACTOR â€” ToF
    # ----------------------------------------------------------
    'estimator__extractor__tof_modes':      [
        'pooled_stats|sensor_stats',
        #   'sensor_stats', 
          'raw',
        #     'pooled_stats'
            ],
    'estimator__extractor__tof_fill_mode':  [
                                            # 'far_255',
                                              'far_500'
                                              ],
    'estimator__extractor__tof_n_sensors':  [None],

    # ----------------------------------------------------------
    # EXTRACTOR â€” thermo
    # ----------------------------------------------------------
    'estimator__extractor__thm_modes': [
        'centered_diff', 
                                        # 'centered', 
                                        # 'diff', 'raw', 'centered|diff'
                                        ],

    # ----------------------------------------------------------
    # EXTRACTOR â€” signal cleaning / motion
    # ----------------------------------------------------------
    'estimator__extractor__motion_filter_mode':     [None],
    'estimator__extractor__use_dead_reckoning':     [False],
    'estimator__extractor__dead_reckoning_detrend': [False],
    'estimator__extractor__kalman_process_noise':       [1e-4],
    'estimator__extractor__kalman_measurement_noise':   [1e-2],
    'estimator__extractor__window_size':   [5],
    'estimator__extractor__smooth_alpha':  [None],
    'estimator__extractor__clip_value':    [None],
    'estimator__extractor__interp_mode':   ['linear'],

    # ----------------------------------------------------------
    # EXTRACTOR â€” chunking
    # ----------------------------------------------------------
    'estimator__extractor__output_format':      ['chunks'],
    'estimator__extractor__padding_value':      [0.0],
    'estimator__extractor__maxlen':             [None],
    'estimator__extractor__sequence_crop_mode': [
        # 'head',
        #   'tail',
        #     'center',
          'none'
          ],
    'estimator__extractor__chunk_window_size':  [250, 300, 350, 400],
    'estimator__extractor__chunk_stride':       [50, 70, 100, 150],
    'estimator__extractor__use_chunk_stride_ratio': [True],
    'estimator__extractor__chunk_stride_ratio':  [0.2, 0.3, 0.8, 0.9],
    'estimator__extractor__final_window_mode':  [
        # 'pad', 
        'overlap'
        ],
    'estimator__extractor__window_cap_policy':  [
        # 'last',
        #   'first',
            'uniform'],
    'estimator__extractor__max_windows_per_sequence': [200],
    'estimator__extractor__add_global_context': [False],
    'estimator__extractor__resample_modalities':[False],
    'estimator__extractor__compute_dt':         [False],

    # ----------------------------------------------------------
    # EXTRACTOR â€” sampling rates
    # ----------------------------------------------------------
    'estimator__extractor__imu_native_sampling_rate': [IMU_ASSUMED_FS_HZ],
    'estimator__extractor__rot_native_sampling_rate': [ROT_ASSUMED_FS_HZ],
    'estimator__extractor__tof_native_sampling_rate': [TOF_ASSUMED_FS_HZ],
    'estimator__extractor__thm_native_sampling_rate': [THERMO_ASSUMED_FS_HZ],
    'estimator__extractor__imu_target_sampling_rate': [IMU_ASSUMED_FS_HZ],
    'estimator__extractor__rot_target_sampling_rate': [ROT_ASSUMED_FS_HZ],
    'estimator__extractor__tof_target_sampling_rate': [TOF_ASSUMED_FS_HZ],
    'estimator__extractor__thm_target_sampling_rate': [THERMO_ASSUMED_FS_HZ],

    # ----------------------------------------------------------
    # EXTRACTOR â€” STFT/CWT
    # ----------------------------------------------------------
    'estimator__extractor__stft_configs':       STFT_CONFIGS,
    'estimator__extractor__cwt_configs':       CWT_CONFIGS,

    # ----------------------------------------------------------
    # EXTRACTOR â€” problematic-sequence filter
    # ----------------------------------------------------------
    'estimator__extractor__filter_problematic_sequences':   [problematic_sequence_bool],
    'estimator__extractor__ideal_skew_threshold':           [1.7],
    'estimator__extractor__problematic_features_threshold': [7],
    'estimator__extractor__problematic_feature_cols': [
        # None,
        ['acc_x', 'acc_y', 'acc_z', 'rot_x', 'rot_y', 'rot_w', 'rot_z'],
    ],

    # ----------------------------------------------------------
    # MODEL â€” branch architecture
    # ----------------------------------------------------------
    'estimator__branch_filters':          branch_filter_candidates,
    'estimator__branch_num_conv_layers':  branch_depth_candidates,
    'estimator__branch_filter_mode':      [branch_filter_mode],
    'estimator__branch_kernel_sizes': [
        # {'acc': 3,  'rotation': 3,  'tof': 3, 'thermo': 3, 'stft': 3, 'cwt': 3},
        # {'acc': 5,  'rotation': 5,  'tof': 3, 'thermo': 3, 'stft': 3, 'cwt': 3},
        # {'acc': 7,  'rotation': 7,  'tof': 5, 'thermo': 3, 'stft': 3, 'cwt': 3},
        {'acc': 7,  'rotation': 7,  'tof': 7, 'thermo': 3, 'stft': 3, 'cwt': 3},
        {'acc': 8, 'rotation': 8, 'tof': 8, 'thermo': 3, 'stft': 3, 'cwt': 3},
    ],
    'estimator__branch_pool_sizes': [
        {'acc': 1, 'rotation': 1, 'tof': 1, 'thermo': 1, 'stft': 1, 'cwt': 1},
        # {'acc': 2, 'rotation': 2, 'tof': 2, 'thermo': 1, 'stft': 1, 'cwt': 1},
        # {'acc': 3, 'rotation': 3, 'tof': 3, 'thermo': 2, 'stft': 2, 'cwt': 2},
        # {'acc': 4, 'rotation': 4, 'tof': 4, 'thermo': 2, 'stft': 2, 'cwt': 2},
        # {'acc': 5, 'rotation': 5, 'tof': 5, 'thermo': 2, 'stft': 2, 'cwt': 2},
    ],
    'estimator__branch_dense_units':  [225],
    'estimator__static_branch_units':  [100],

    # ----------------------------------------------------------
    # MODEL â€” temporal/fusion architecture
    # ----------------------------------------------------------
    # Commented alternatives: replace existing keys; do not add duplicate keys.
    # Bayesian search inherits these options from GRID_PARAM_SPACE.
    'estimator__temporal_architecture': [
        'bert'
    ], #  'bigru_attention', 'phase_attention', 'cnn_attention_pool', 'bert'
    'estimator__use_bigru': [False],
    'estimator__gru_units': [32],
    'estimator__gru_layers': [1],
    'estimator__gru_dropout': [0.0],
    'estimator__gru_recurrent_dropout': [0.0],
    'estimator__bidirectional_gru': [True],
    'estimator__attention_type': ['additive', 'linear'],
    'estimator__attention_units': [64],
    'estimator__cnn_depth': [3, 4],
    'estimator__cnn_filters': [32, 64, 96],
    'estimator__cnn_kernel_size': [5, 7],
    'estimator__cnn_dropout': [0.0],
    'estimator__use_residual': [False],
    'estimator__use_se': [False],
    'estimator__se_ratio': [4],
    'estimator__n_phases': [3],
    'estimator__phase_attention_units': [16],
    'estimator__phase_attention_temperature': [0.75],
    'estimator__phase_fusion_mode': ['concat'], # concat, sum, learned
    # cnn is an alias for conv1d.
    'estimator__branch_encoder_type': ['conv1d', 'cnn'],
    'estimator__branch_cnn_filters': [16, 32, 64],
    'estimator__branch_cnn_kernel_size': [5, 6, 7],
    'estimator__branch_cnn_depth': [3, 4],
    'estimator__branch_dropout': [0.0],
    'estimator__post_fusion_attention_type': ['additive'], # additive, linear
    'estimator__post_fusion_attention_units': [64],
    'estimator__post_fusion_pooling': ['attention'], # 'attention', 'avg', 'max', 'avg_max', 'attention_avg_max'
    'estimator__bert_num_layers': [3, 4, 5],
    'estimator__bert_d_model': [32, 64, 128, 200, 256],
    'estimator__bert_num_heads': [4, 5, 6],
    'estimator__bert_ff_dim': [256, 300],
    'estimator__bert_dropout': [0.0],
    'estimator__bert_attention_dropout': [0.0],
    'estimator__bert_activation': ['gelu', 'relu'], # gelu, relu
    'estimator__bert_pooling': ['attention', 'avg', 'avg_max', 'cls'], # attention, avg , avg_max
    # CLS pooling requires bert_use_cls_token=True; replace both entries together.
    'estimator__bert_use_cls_token': [True],
    # Enable the BERT CLS token.

    'estimator__bert_positional_encoding': ['sinusoidal', 'learned'], # learned, sinusoidal
    # 'estimator__tof_encoder_type': ['flatten_1d'],
    # Spatial ToF requires extractor__tof_modes='raw' (8x8 pixel groups).
    'estimator__tof_encoder_type': ['flatten_1d'],
    'estimator__tof_spatial_filters': [8],
    'estimator__tof_spatial_kernel_size': [3],
    'estimator__tof_spatial_depth': [1],
    'estimator__tof_gru_units': [16],
    'estimator__fusion_type': ['attention'],
    'estimator__fusion_units': [128],
    'estimator__fusion_dropout': [0.0],
    'estimator__use_orientation_aux_head': [USE_ORIENTATION_AUX_HEAD],
    # 'estimator__orientation_num_classes': [4],  # Inferred from orientation labels during fit.
    # Requires y with an 'orientation' column instead of the current 1-D target.
    # 'estimator__use_orientation_aux_head': [True],
    'estimator__use_phase_aux_head': [USE_PHASE_AUX_HEAD],
    'estimator__custom_aux_heads': [NO_CUSTOM_AUX_HEADS],
    'estimator__orientation_loss_weight': [ORIENTATION_LOSS_WEIGHT],
    'estimator__phase_loss_weight': [PHASE_LOSS_WEIGHT],
    # Optional replacements to search head combinations and strengths:
    # 'estimator__custom_aux_heads': [[], CUSTOM_AUX_HEADS],
    # 'estimator__orientation_loss_weight': [0.1, 0.2, 0.5],
    # 'estimator__phase_loss_weight': [0.1, 0.2, 0.5],
    # Requires PHASE_LABEL_PROVIDER; otherwise this candidate receives the penalty.
    # 'estimator__use_phase_aux_head': [True],

    # ----------------------------------------------------------
    # MODEL â€” head, regularization, activation, pooling
    # ----------------------------------------------------------
    'estimator__conv_dropout':    [0.0],
    'estimator__static_dropout':  [0.0],
    'estimator__dense_units':     [128, 200, 256, 300],
    'estimator__dropout':         [0.0],
    'estimator__use_batchnorm':   [True],
    'estimator__activation':      [
        'relu', 
                                #    'gelu', 
                                #    'swish',
                                    # 'elu',
                                    #   'leaky'
                                      ],
    'estimator__pooling_mode':    [
        'attention'
                                #    , 'mean'
                                   ],

    # ----------------------------------------------------------
    # MODEL â€” optimizer
    # ----------------------------------------------------------
    'estimator__optimizer_name':      [
                                        # 'adam', 
                                        'adamw', 
                                        # 'rmsprop', 
                                    #    'sgd', 
                                    #    'nadam',
                                        #  'adagrad', 'adadelta'
                                         ],
    'estimator__learning_rate':       [
        1e-5,
        5e-5,
        1e-4,
        #   1e-3
        ],
    'estimator__weight_decay':        [0.0], # adam
    'estimator__momentum':            [0.0], # sgd
    'estimator__gradient_clip_norm':  [None],
    'estimator__gradient_clip_value': [None],

    # ----------------------------------------------------------
    # MODEL â€” loss
    # ----------------------------------------------------------
    'estimator__loss_name':       ['sparse_cce', 'focal'], # sparse_cce, focal
    'estimator__label_smoothing': [0.0],

    # ----------------------------------------------------------
    # TRAINING LOOP
    # ----------------------------------------------------------
    'estimator__epochs':                  [150],
    'estimator__batch_size':              [128],
    'estimator__validation_split':        [0.15],
    'estimator__early_stopping_patience': [20],
    'estimator__class_weight_mode':       ['balanced'],

    # ----------------------------------------------------------
    # LR SCHEDULER
    # ----------------------------------------------------------
    'estimator__use_lr_scheduler':   [False],
    # 'estimator__lr_scheduler_type':  ['plateau', 'cosine'],
    # 'estimator__lr_factor':          [0.3, 0.5, 0.7],
    # 'estimator__lr_patience':        [3, 5, 10],
    # 'estimator__min_lr':             [1e-7, 1e-6, 1e-5],
    # 'estimator__cosine_t_max':       [50, 100, 200],
    # 'estimator__warmup_epochs':      [0, 3, 5],
}


# ============================================================
# SEPARATE EVOLUTIONARY PARAM SPACE
# Includes Grid/Bayesian candidates while retaining existing evolutionary choices.
# Numeric ranges remain ranges and expand only when needed for coverage.
# ============================================================

EVOLUTIONARY_PARAM_SPACE = {

    # ----------------------------------------------------------
    # AUGMENTOR (disabled; applied only inside estimator.fit)
    # ----------------------------------------------------------
    'estimator__augmentor__prob':                ECat([0.0]),
    'estimator__augmentor__per_aug_prob':        ECat([0.0]),
    'estimator__augmentor__augmentations':       ECat([None]),
    'estimator__augmentor__sensor_drop_prob':    ECat([0.0]),
    'estimator__augmentor__noise_std':           ECat([0.0]),
    'estimator__augmentor__jitter_sigma':        ECat([0.0]),
    'estimator__augmentor__scaling_sigma':       ECat([0.0]),
    'estimator__augmentor__channel_drop_prob':   ECat([0.0]),
    'estimator__augmentor__time_shift_frac':     ECat([0.0]),
    'estimator__augmentor__temporal_num_masks':  ECat([0]),
    'estimator__augmentor__temporal_mask_frac':  ECat([0.0]),
    'estimator__augmentor__warp_sigma':          ECat([0.0]),
    'estimator__augmentor__warp_num_knots':      ECat([0]),
    'estimator__augmentor__crop_frac_range':     ECat([(1.0, 1.0)]),

    # ----------------------------------------------------------
    # EXTRACTOR — accelerometer
    # ----------------------------------------------------------
    'estimator__extractor__acc_modes': ECat([
        'smoothed|velocity|displacement|jerk',
    ]),
    'estimator__extractor__use_acc_magnitude':         ECat([ True]),
    'estimator__extractor__use_linear_acc_magnitude':  ECat([True]),
    'estimator__extractor__linear_acc_mode':           ECat(['baseline']),
    'estimator__extractor__linear_acc_modes': ECat([
        # '', 'raw', 'velocity', 'displacement', 'jerk',
        'raw|velocity|displacement|jerk',
    ]),
    'estimator__extractor__linear_acc_frame': ECat(['world', 'body']),
    'estimator__extractor__use_highpass_fallback':     ECat([False, True]),

    # ----------------------------------------------------------
    # EXTRACTOR — rotation
    # ----------------------------------------------------------
    'estimator__extractor__rotation_modes': ECat([
        'quaternion|euler|angular_velocity',
    ]),
    'estimator__extractor__fix_quaternion_sign': ECat([True]),

    # ----------------------------------------------------------
    # EXTRACTOR — ToF
    # ----------------------------------------------------------
    'estimator__extractor__tof_modes':      ECat(['pooled_stats|sensor_stats', 'raw']),
    'estimator__extractor__tof_fill_mode':  ECat(['far_500']),
    'estimator__extractor__tof_n_sensors':  ECat([None]),

    # ----------------------------------------------------------
    # EXTRACTOR — thermo
    # ----------------------------------------------------------
    'estimator__extractor__thm_modes': ECat(['centered_diff']),

    # ----------------------------------------------------------
    # EXTRACTOR — signal cleaning / motion
    # ----------------------------------------------------------
    'estimator__extractor__motion_filter_mode':         ECat([None]),
    'estimator__extractor__use_dead_reckoning':         ECat([False]),
    'estimator__extractor__dead_reckoning_detrend':     ECat([False]),
    'estimator__extractor__kalman_process_noise':       ECat([1e-4]),
    'estimator__extractor__kalman_measurement_noise':   ECat([1e-2]),
    'estimator__extractor__window_size':                ECat([5]),
    'estimator__extractor__smooth_alpha':               ECat([None]),
    'estimator__extractor__clip_value':                 ECat([None]),
    'estimator__extractor__interp_mode':                ECat(['linear']),

    # ----------------------------------------------------------
    # EXTRACTOR — chunking
    # ----------------------------------------------------------
    'estimator__extractor__output_format':            ECat(['chunks']),
    'estimator__extractor__padding_value':            ECat([0.0]),
    'estimator__extractor__maxlen':                   ECat([None]),
    'estimator__extractor__sequence_crop_mode':       ECat(['none']),
    'estimator__extractor__chunk_window_size':        ECat([100, 150, 200, 250, 300, 350, 400]),
    'estimator__extractor__chunk_stride':             ECat([50, 70, 100, 150]),
    'estimator__extractor__use_chunk_stride_ratio':   ECat([True]),
    'estimator__extractor__chunk_stride_ratio':       EFloat(0.2, 0.9),
    'estimator__extractor__final_window_mode':        ECat(['overlap']),
    'estimator__extractor__window_cap_policy':        ECat(['uniform']),
    'estimator__extractor__max_windows_per_sequence': ECat([150, 200]),
    'estimator__extractor__add_global_context':       ECat([False]),
    'estimator__extractor__frame_stats': ECat([
        'mean,std,min,max,last',
        'mean,std,min,max,first,last,median,rms,abs_mean',
        'mean,std,median,rms',
    ]),
    'estimator__extractor__resample_modalities':      ECat([False]),
    'estimator__extractor__compute_dt':               ECat([False]),

    # ----------------------------------------------------------
    # EXTRACTOR — sampling rates
    # ----------------------------------------------------------
    'estimator__extractor__imu_native_sampling_rate': ECat([IMU_ASSUMED_FS_HZ]),
    'estimator__extractor__rot_native_sampling_rate': ECat([ROT_ASSUMED_FS_HZ]),
    'estimator__extractor__tof_native_sampling_rate': ECat([TOF_ASSUMED_FS_HZ]),
    'estimator__extractor__thm_native_sampling_rate': ECat([THERMO_ASSUMED_FS_HZ]),
    'estimator__extractor__imu_target_sampling_rate': ECat([IMU_ASSUMED_FS_HZ]),
    'estimator__extractor__rot_target_sampling_rate': ECat([ROT_ASSUMED_FS_HZ]),
    'estimator__extractor__tof_target_sampling_rate': ECat([TOF_ASSUMED_FS_HZ]),
    'estimator__extractor__thm_target_sampling_rate': ECat([THERMO_ASSUMED_FS_HZ]),

    # ----------------------------------------------------------
    # EXTRACTOR — STFT/CWT
    # ----------------------------------------------------------
    # [{}] inherits the scalar settings below; named configs can override them.
    # Preserve every existing config-list candidate.
    'estimator__extractor__stft_configs': ECat(STFT_CONFIGS),
    'estimator__extractor__cwt_configs':  ECat(CWT_CONFIGS),
    # 'estimator__extractor__stft_nperseg': ECat([8, 16, 32, 64, 128]),
    # 'estimator__extractor__stft_noverlap': ECat([None, 0, 4]),
    # 'estimator__extractor__stft_window_seconds': ECat([None, 0.5, 1.0, 2.0]),
    # 'estimator__extractor__stft_window_type': ECat(['hann', 'hamming', 'blackman', 'boxcar']),
    # 'estimator__extractor__stft_scaling': ECat(['density', 'spectrum', 'magnitude']),
    # 'estimator__extractor__stft_detrend': ECat(['constant', 'linear', False]),
    # 'estimator__extractor__stft_use_log_scale': ECat([True, False]),
    # 'estimator__extractor__stft_frequency_bands': ECat([
    #     None,
    #     {'low': (0.0, 3.0), 'mid': (3.0, 6.0), 'high': (6.0, 10.0)},
    # ]),
    # 'estimator__extractor__cwt_wavelet': ECat(['morl', 'mexh', 'gaus1', 'cmor1.5-1.0']),
    # 'estimator__extractor__cwt_max_scale': EInt(16, 128),
    # 'estimator__extractor__cwt_n_scales': EInt(8, 32),
    # 'estimator__extractor__cwt_use_log_scale': ECat([True, False]),
    # 'estimator__extractor__cwt_min_scale': ECat([None, 4.0, 8.0]),
    # 'estimator__extractor__cwt_max_frequency': ECat([None, 10.0, 20.0]),

    # ----------------------------------------------------------
    # EXTRACTOR — problematic-sequence filter
    # ----------------------------------------------------------
    'estimator__extractor__filter_problematic_sequences':   ECat([problematic_sequence_bool]),
    'estimator__extractor__ideal_skew_threshold':           ECat([1.7]),
    'estimator__extractor__problematic_features_threshold': ECat([7]),
    'estimator__extractor__problematic_feature_cols': ECat([
        ['acc_x', 'acc_y', 'acc_z', 'rot_x', 'rot_y', 'rot_w', 'rot_z'],
    ]),

    # ----------------------------------------------------------
    # MODEL — branch architecture
    # ----------------------------------------------------------
    'estimator__branch_filters':          ECat(branch_filter_candidates),
    'estimator__branch_num_conv_layers':  ECat(branch_depth_candidates),
    'estimator__branch_filter_mode':      ECat([branch_filter_mode]),
    'estimator__branch_kernel_sizes': ECat([
        {'acc': 7, 'rotation': 7, 'tof': 7, 'thermo': 3, 'stft': 3, 'cwt': 3},
        {'acc': 8, 'rotation': 8, 'tof': 8, 'thermo': 3, 'stft': 3, 'cwt': 3},
    ]),
    'estimator__branch_pool_sizes': ECat([
        {'acc': 1, 'rotation': 1, 'tof': 1, 'thermo': 1, 'stft': 1, 'cwt': 1},
    ]),
    'estimator__branch_dense_units':  ECat([225]),
    'estimator__static_branch_units': ECat([100]),

    # ----------------------------------------------------------
    # MODEL — temporal/fusion architecture
    # ----------------------------------------------------------
    # Commented alternatives: replace existing keys; do not add duplicate keys.
    'estimator__temporal_architecture': ECat([
'bert',
    ]),
    'estimator__use_bigru': ECat([False]),
    # 'estimator__gru_units': ECat([32, 64]),
    # 'estimator__gru_layers': EInt(1, 2),
    # 'estimator__gru_dropout': EFloat(0.0, 0.1),
    # 'estimator__gru_recurrent_dropout': ECat([0.0]),
    # 'estimator__bidirectional_gru': ECat([True, False]),
    'estimator__attention_type': ECat(['additive', 'linear']),
    'estimator__attention_units': ECat([32, 64, 100]),
    'estimator__cnn_depth': ECat([1, 2, 3, 4, 5]),
    'estimator__cnn_filters': ECat([32, 64, 96]),
    'estimator__cnn_kernel_size': ECat([3, 5, 7]),
    'estimator__cnn_dropout': ECat([0.0, 0.1]),
    'estimator__use_residual': ECat([False, True]),
    'estimator__use_se': ECat([False, True]),
    'estimator__se_ratio': ECat([4, 8, 16]),
    'estimator__n_phases': ECat([3, 4]),
    'estimator__phase_attention_units': ECat([16, 32, 64]),
    'estimator__phase_attention_temperature': ECat([0.75, 1.0, 1.5]),
    'estimator__phase_fusion_mode': ECat(['concat', 'sum', 'learned']),
    # cnn is an alias for conv1d.
    'estimator__branch_encoder_type': ECat(['conv1d', 'cnn']),
    'estimator__branch_cnn_filters': ECat([16, 32, 64, 128]),
    'estimator__branch_cnn_kernel_size': ECat([3, 5, 6, 7]),
    'estimator__branch_cnn_depth': EInt(1, 4),
    'estimator__branch_dropout': EFloat(0.0, 0.4),
    'estimator__post_fusion_attention_type': ECat(['additive', 'linear']),
    'estimator__post_fusion_attention_units': ECat([32, 64]),
    'estimator__post_fusion_pooling': ECat(['attention', 'avg', 'max', 'avg_max', 'attention_avg_max']),
    'estimator__bert_num_layers': EInt(1, 5),
    'estimator__bert_d_model': ECat([32, 64, 128, 200, 256]),
    'estimator__bert_num_heads': ECat([2, 4, 5, 6]),
    'estimator__bert_ff_dim': ECat([64, 128, 256, 300]),
    'estimator__bert_dropout': ECat([0.0]),
    'estimator__bert_attention_dropout': ECat([0.0]),
    'estimator__bert_activation': ECat(['gelu', 'relu']),
    'estimator__bert_pooling': ECat(['attention', 'avg', 'avg_max', 'cls']),
    # Candidate rules require bert_use_cls_token=True for CLS pooling.
    'estimator__bert_use_cls_token': ECat([False, True]),
    'estimator__bert_positional_encoding': ECat(['learned', 'sinusoidal']),
    # Spatial ToF requires extractor__tof_modes='raw' (8x8 pixel groups).
    'estimator__tof_encoder_type': ECat(['flatten_1d']),
    # 'estimator__tof_spatial_filters': ECat([8, 16, 32]),
    # 'estimator__tof_spatial_kernel_size': ECat([3, 5]),
    # 'estimator__tof_spatial_depth': ECat([1, 2]),
    # 'estimator__tof_gru_units': ECat([16, 32, 64]),
    'estimator__fusion_type': ECat(['concat', 'sum', 'attention']),
    'estimator__fusion_units': ECat([32, 64, 128]),
    'estimator__fusion_dropout': EFloat(0.0, 0.1),
    'estimator__use_orientation_aux_head': ECat([USE_ORIENTATION_AUX_HEAD]),
    # 'estimator__orientation_num_classes': ECat([4]),  # Inferred from orientation labels during fit.
    # Requires y with an 'orientation' column instead of the current 1-D target.
    # 'estimator__use_orientation_aux_head': ECat([True]),
    'estimator__use_phase_aux_head': ECat([USE_PHASE_AUX_HEAD]),
    'estimator__custom_aux_heads': ECat([NO_CUSTOM_AUX_HEADS, generic_aux_heads, competition_custom_aux_heads, multi_scale_custom_aux_heads, qwen_custom_aux_heads]),
    # 'estimator__orientation_loss_weight': ECat(list(dict.fromkeys([ORIENTATION_LOSS_WEIGHT, 0.1, 0.2, 0.5]))),
    # 'estimator__phase_loss_weight': ECat(list(dict.fromkeys([PHASE_LOSS_WEIGHT, 0.1, 0.2, 0.5]))),
    # Requires PHASE_LABEL_PROVIDER; otherwise this candidate receives the penalty.
    # 'estimator__use_phase_aux_head': ECat([True]),  # Requires phase labels.

    # ----------------------------------------------------------
    # MODEL — head, regularization, activation, pooling
    # ----------------------------------------------------------
    'estimator__conv_dropout':    ECat([0.0]),
    'estimator__static_dropout':  ECat([0.0]),
    'estimator__dense_units':     ECat([128, 200, 256, 300]),
    'estimator__dropout':         ECat([0.0]),
    'estimator__use_batchnorm':   ECat([True]),
    'estimator__activation':      ECat(['relu']),
    'estimator__pooling_mode':    ECat(['attention']),

    # ----------------------------------------------------------
    # MODEL — optimizer
    # ----------------------------------------------------------
    'estimator__optimizer_name':      ECat(['adamw']),
    'estimator__learning_rate':       EFloat(1e-5, 1e-4, distribution='log-uniform'),
    'estimator__weight_decay':        ECat([0.0]),
    'estimator__momentum':            ECat([0.0]),
    'estimator__gradient_clip_norm':  ECat([None]),
    'estimator__gradient_clip_value': ECat([None]),

    # ----------------------------------------------------------
    # MODEL — loss
    # ----------------------------------------------------------
    'estimator__loss_name':       ECat(['sparse_cce', 'focal']),
    'estimator__label_smoothing': ECat([0.0]),

    # ----------------------------------------------------------
    # TRAINING LOOP
    # ----------------------------------------------------------
    'estimator__epochs':                  ECat([100, 150]),
    'estimator__batch_size':              ECat([128]),
    'estimator__validation_split':        ECat([0.15]),
    'estimator__early_stopping_patience': ECat([20]),
    'estimator__class_weight_mode':       ECat(['balanced']),

    # ----------------------------------------------------------
    # LR SCHEDULER
    # ----------------------------------------------------------
    'estimator__use_lr_scheduler':   ECat([False]),
    # 'estimator__lr_scheduler_type':  ECat(['plateau', 'cosine']),
    # 'estimator__lr_factor':          EFloat(0.3, 0.7),
    # 'estimator__lr_patience':        EInt(3, 10),
    # 'estimator__min_lr':             EFloat(1e-7, 1e-5, distribution='log-uniform'),
    # 'estimator__cosine_t_max':       EInt(50, 200),
    # 'estimator__warmup_epochs':      EInt(0, 5),
}


# ============================================================
# Fixed data/runtime context: expose each remaining leaf parameter without
# changing labels, acquisition metadata, reproducibility, or failure handling.
# Add a key to the dictionary above to replace its fixed choice explicitly.
# orientation_num_classes is inferred from the training labels during fit.
# ============================================================

EVOLUTIONARY_FIXED_PARAM_NAMES = (
    'estimator__extractor__acceleration_units',
    'estimator__extractor__gravity_magnitude',
    'estimator__extractor__sequence_col',
    'estimator__extractor__counter_col',
    'estimator__augmentor__sequence_col',
    'estimator__augmentor__counter_col',
    'estimator__augmentor__seed',
    'estimator__primary_target',
    'estimator__orientation_num_classes',
    'estimator__phase_label_provider',
    'estimator__random_state',
    'estimator__verbose',
    'estimator__invalid_candidate_policy',
    'estimator__invalid_candidate_score',
    'estimator__candidate_failure_log',
)
_pipeline_params = pipeline.get_params(deep=True)
for _key in EVOLUTIONARY_FIXED_PARAM_NAMES:
    EVOLUTIONARY_PARAM_SPACE.setdefault(_key, ECat([_pipeline_params[_key]]))

# Component objects and Pipeline plumbing are not individual hyperparameters.
_evolutionary_leaf_params = {
    key for key in _pipeline_params
    if key.startswith('estimator__')
    and key not in {'estimator__extractor', 'estimator__augmentor'}
}
_missing_evolutionary_params = _evolutionary_leaf_params - set(EVOLUTIONARY_PARAM_SPACE)
# Omitted search dimensions inherit the configured estimator defaults.


# ============================================================
# Build and validate
# ============================================================

if SKOPT_AVAILABLE:
    BAYESIAN_PARAM_SPACE = multibranch_keras.prepare_multibranch_bayesian_space(
        GRID_PARAM_SPACE
    )
else:
    BAYESIAN_PARAM_SPACE = GRID_PARAM_SPACE

EVOLUTIONARY_SEARCH_SPACE = multibranch_keras.prepare_multibranch_evolutionary_space(
    EVOLUTIONARY_PARAM_SPACE
)

for space_name, candidate_space in (
    ('grid', GRID_PARAM_SPACE),
    ('bayesian', BAYESIAN_PARAM_SPACE),
    ('evolutionary', EVOLUTIONARY_SEARCH_SPACE),
):
    unknown_params = set(candidate_space) - set(pipeline.get_params(deep=True))
    if unknown_params:
        raise ValueError(
            f'Unknown {space_name} search parameters: {sorted(unknown_params)}'
        )

if search_mode == 'bayes':
    param_space = BAYESIAN_PARAM_SPACE
elif search_mode == 'evo':
    param_space = EVOLUTIONARY_SEARCH_SPACE
else:
    param_space = GRID_PARAM_SPACE

print(
    f'Validated {len(GRID_PARAM_SPACE)} Grid/Bayesian and '
    f'{len(EVOLUTIONARY_PARAM_SPACE)} Evolutionary search parameters.'
)
print(f'Total reachable params in pipeline: {len(pipeline.get_params(deep=True))}')

# Train

In [ ]:
if search_mode == 'bayes':
    if not SKOPT_AVAILABLE:
        raise ImportError(
            "Bayesian search requires scikit-optimize. "
            "Install with: pip install scikit-optimize"
        )
    search = HierarchicalBayesSearchCV(
        estimator=pipeline,
        search_spaces=param_space,
        n_iter=n_iter,
        cv=cv_object,
        n_jobs=n_jobs,
        random_state=random_state,
        verbose=verbose,
        return_train_score=True,
        scoring=candidate_score,
        refit=False,
        error_score=error_score,
    )
elif search_mode == 'evo':
    search = HierarchicalGASearchCV(
        estimator=pipeline,
        param_grid=param_space,
        cv=cv_object,
        population_size=population_size,
        generations=generations,
        tournament_size=tournament_size,
        elitism=elitism,
        crossover_probability=crossover_probability,
        mutation_probability=mutation_probability,
        random_state=random_state,
        verbose=verbose,
        return_train_score=True,
        scoring=candidate_score,
        refit=False,
        error_score=error_score,
    )
elif search_mode == 'best params':
    unknown = set(BEST_PARAMS) - set(pipeline.get_params(deep=True))
    if unknown:
        raise ValueError(f'Unsupported best parameters: {sorted(unknown)}')
    # One candidate: measure it on the current CV split and refit for holdout.
    # Preserve saved values even when outside the current search ranges.
    search = GridSearchCV(
        estimator=pipeline,
        param_grid={key: [value] for key, value in BEST_PARAMS.items()},
        cv=cv_object,
        n_jobs=n_jobs,
        verbose=verbose,
        return_train_score=True,
        scoring=candidate_score,
        refit=False,
        error_score=error_score,
    )
else:
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_space,
        cv=cv_object,
        n_jobs=n_jobs,
        verbose=verbose,
        return_train_score=True,
        scoring=candidate_score,
        refit=False,
        error_score=error_score,
    )

search.fit(X_train, y_train, groups=groups)

experiment_succeeded = refit_valid_candidate(search, X_train, y_train)
# Keep the standard result-column names used by the existing reporting code.
pd.DataFrame(search.cv_results_).to_csv(results_dir / f'{timestamp}_raw_cv.csv', index=False)
with open(results_dir / f'{timestamp}_refit_failures.json', 'w') as stream:
    json.dump(search.refit_errors_, stream, indent=2)
if experiment_succeeded:
    print('Best CV score:', search.best_score_)
    print('Best params:', search.best_params_)
else:
    print('No valid candidate could be fitted. CV scores and failure logs saved; holdout skipped.')

# Analysis and save

In [ ]:
if experiment_succeeded:
    best_model = search.best_estimator_
    save_dir = results_dir / timestamp
    save_dir.mkdir(parents=True, exist_ok=True)

    # --- pull the history out of the fitted pipeline ---
    best_estimator = best_model.named_steps['estimator']
    history = best_estimator.get_history_dict()   # returns self.history_

    loss      = history.get('loss', [])
    val_loss  = history.get('val_loss', [])
    acc       = history.get('accuracy', history.get('predictions_accuracy', []))
    val_acc   = history.get('val_accuracy', history.get('val_predictions_accuracy', []))

    has_val = len(val_loss) > 0 and len(val_acc) > 0
    epochs_ran = range(1, len(loss) + 1)

    # --- two subplots, two lines each ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ---- Loss ----
    ax = axes[0]
    ax.plot(epochs_ran, loss, label='train loss', linewidth=2)
    if has_val:
        ax.plot(epochs_ran, val_loss, label='val loss', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training vs Validation Loss')
    ax.grid(alpha=0.3)
    ax.legend()

    # ---- Accuracy ----
    ax = axes[1]
    ax.plot(epochs_ran, acc, label='train acc', linewidth=2)
    if has_val:
        ax.plot(epochs_ran, val_acc, label='val acc', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.set_title('Training vs Validation Accuracy')
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()

    # save next to the other results (save_dir is defined in your last cell)
    curves_path = save_dir / f'{experiment_name}_curves.png'
    # plt.savefig(curves_path, dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Saved curves to {curves_path}')
    print(f'Epochs run: {len(loss)}')
    if has_val:
        best_epoch = int(np.argmin(val_loss)) + 1
        print(f'Best val_loss epoch: {best_epoch}  (val_loss={min(val_loss):.4f})')

In [ ]:
if experiment_succeeded:
    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test)

    eval_results = evaluate_holdout(
        y_test,
        y_pred,
        target_col=TARGET_COL,
        verbose=True,
    )

    print('Holdout competition score / macro F1:', eval_results['competition_score'])

In [ ]:
save_dir = results_dir / timestamp
save_dir.mkdir(parents=True, exist_ok=True)

cv_results_with_summary_df = (
    build_quick_results_df(search, holdout_score=eval_results['competition_score'])
    if experiment_succeeded else pd.DataFrame(search.cv_results_)
)

if quick_save or not experiment_succeeded:
    quick_csv_path = save_dir / f'{experiment_name}_cv.csv'
    cv_results_with_summary_df.to_csv(
        quick_csv_path,
        index=False,
    )
    print(f'Quick results saved to {quick_csv_path}')
else:
    best_estimator = best_model.named_steps['estimator']
    best_estimator.model_.save(save_dir / 'best_model.h5')

    cv_results_with_summary_df.to_csv(
        save_dir / f'{experiment_name}_cv.csv',
        index=False,
    )
    eval_results['results_df'].to_csv(
        save_dir / f'{experiment_name}_holdout.csv',
        index=False,
    )
    pd.DataFrame([{
        'target': TARGET_COL,
        'best_score': search.best_score_,
        'best_params': json.dumps(search.best_params_, default=str),
        'holdout_score': eval_results['competition_score'],
    }]).to_csv(
        save_dir / f'{experiment_name}_best.csv',
        index=False,
    )

    with open(save_dir / 'results.json', 'w') as results_file:
        json.dump({
            'target': TARGET_COL,
            'best_score': float(search.best_score_),
            'best_params': search.best_params_,
            'holdout_score': float(eval_results['competition_score']),
            'random_state': random_state,
            'train_sequences': int(X_train['sequence_id'].nunique()),
            'holdout_sequences': int(X_test['sequence_id'].nunique()),
        }, results_file, indent=2, default=str)

    print(f'Results saved to {save_dir}')